# TRUST-MANET dataset — Kaggle runner

Self-contained Kaggle notebook: it **writes all source files itself** (nothing to upload),
builds ns-3, runs the simulation matrix, and produces the Parquet dataset.

### Kaggle setup (do this first)
1. **Turn Internet ON** — right panel → *Settings* → *Internet* → **On**
   (needs a phone-verified Kaggle account). Without it, `git`/`apt`/`pip` fail.
2. Accelerator: **None (CPU)** is fine — no GPU needed.
3. For long jobs use **Save Version → Save & Run All (Commit)** — it runs in the
   background up to 12 h even if you close the tab.

### Storage on Kaggle (no Google Drive)
- `/kaggle/working/` → **saved** as notebook output, but capped at ~20 GB and ~500 files.
- `/kaggle/temp/` → fast scratch, **wiped** when the session ends.

So the heavy ns-3 build and the many raw log files go in `/kaggle/temp`, and the
final (small) dataset goes in `/kaggle/working`.

### Time estimate (CPU, ~4 cores)
- ns-3 build ~8–15 min, screening matrix (36 runs) + dataset ~20–40 min → **~30–55 min total**.
- Full matrix is 450 runs — run it in chunks (see the last cell).

## 1. Paths (Kaggle) — no Drive mount needed

In [1]:
import os
CODE_DIR  = '/kaggle/working/trust-manet-dataset'   # source files (saved)
NS3_DIR   = '/kaggle/temp/ns-3-dev'                 # heavy build (scratch, not saved)
RAW_DIR   = '/kaggle/temp/raw'                      # many files (scratch; avoids 500-file cap)
DS_DIR    = '/kaggle/working/dataset'               # final Parquet (saved)
SPLIT_DIR = '/kaggle/working/dataset_split'         # split dataset (saved)

for d in (CODE_DIR, RAW_DIR, DS_DIR, SPLIT_DIR,
          f'{CODE_DIR}/ns3', f'{CODE_DIR}/scripts'):
    os.makedirs(d, exist_ok=True)
os.chdir(CODE_DIR)
print('code dir :', CODE_DIR)
print('ns-3 dir :', NS3_DIR, '(scratch)')
print('outputs  :', DS_DIR, '|', SPLIT_DIR, '(saved)')


code dir : /kaggle/working/trust-manet-dataset
ns-3 dir : /kaggle/temp/ns-3-dev (scratch)
outputs  : /kaggle/working/dataset | /kaggle/working/dataset_split (saved)


## 2. Install build tools + Python deps  (Internet must be ON)

In [2]:
!apt-get -qq update
!apt-get -qq install -y git g++ cmake make python3 >/dev/null
!pip -q install pandas numpy pyarrow pyyaml
import subprocess
print(subprocess.run(['g++','--version'],capture_output=True,text=True).stdout.splitlines()[0])
print(subprocess.run(['cmake','--version'],capture_output=True,text=True).stdout.splitlines()[0])


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
cmake version 3.31.10


## 3. Write the source files

Each cell writes one file with `%%writefile`. Edit + re-run a cell to change code.

In [5]:
%%writefile ns3/malicious-routing.h
/*
 * TRUST-MANET : malicious-routing.h
 * ---------------------------------
 * A thin subclass of ns3::aodv::RoutingProtocol that injects forwarding
 * misbehaviour for a selected set of "compromised" nodes:
 *
 *   BLACKHOLE  -> drop ALL transit (forwarded) application data packets.
 *   GRAYHOLE   -> drop transit application data packets with probability p.
 *   NONE       -> behave exactly like stock AODV (benign node).
 *
 * Only the *data-forwarding* path is altered. All AODV control logic
 * (RREQ/RREP/RERR, route table, local delivery) is inherited unchanged,
 * so routing still converges and the misbehaviour is observable purely as
 * a forwarding anomaly -- which is exactly the signal the local IDS in the
 * TRUST-MANET proposal is meant to detect, separated from mobility loss.
 *
 * NOTE on attack *attraction*: a full blackhole also spoofs RREPs (high dst
 * seqno, hop-count 1) to attract traffic. That requires editing AODV's
 * RecvRequest/SendReply and is version-specific. This file deliberately
 * implements the lower-risk, version-stable *drop* behaviour only. See
 * README.md "Optional: RREP-spoofing attraction" for how to add it.
 *
 * Targeted ns-3 version: ns-3.42 (public Ipv4RoutingProtocol / aodv API).
 */
#ifndef TRUST_MANET_MALICIOUS_ROUTING_H
#define TRUST_MANET_MALICIOUS_ROUTING_H

#include "ns3/aodv-routing-protocol.h"
#include "ns3/random-variable-stream.h"
#include "ns3/nstime.h"
#include "ns3/ipv4-header.h"
#include "ns3/ipv4-route.h"

namespace ns3
{

class MaliciousRouting : public aodv::RoutingProtocol
{
  public:
    enum AttackType
    {
        NONE = 0,
        BLACKHOLE = 1,
        GRAYHOLE = 2
    };

    static TypeId GetTypeId();
    MaliciousRouting();
    ~MaliciousRouting() override;

    /** Configure this node as an attacker. dropProb only used for GRAYHOLE. */
    void SetAttack(AttackType type, double dropProb, Time start, Time stop);

    /** Overridden forwarding decision. */
    bool RouteInput(Ptr<const Packet> p,
                    const Ipv4Header& header,
                    Ptr<const NetDevice> idev,
                    const UnicastForwardCallback& ucb,
                    const MulticastForwardCallback& mcb,
                    const LocalDeliverCallback& lcb,
                    const ErrorCallback& ecb) override;

    // --- cumulative counters, read (as deltas) once per measurement window ---
    uint64_t m_recvForward = 0;   // transit data packets that reached the forwarding hook
    uint64_t m_fwdOk = 0;         // transit data packets actually forwarded
    uint64_t m_fwdDropAttack = 0; // transit data packets dropped by the attack
    uint64_t m_noRoute = 0;       // ErrorCallback invocations (no route / route break)

    AttackType GetAttackType() const { return m_attack; }

  private:
    // Static hooks are bound (via MakeBoundCallback) with `this` + the original
    // callback, so they match the exact ns-3 callback signatures.
    static void ForwardHook(MaliciousRouting* self,
                            UnicastForwardCallback ucb,
                            Ptr<Ipv4Route> route,
                            Ptr<const Packet> p,
                            const Ipv4Header& header);

    static void ErrorHook(MaliciousRouting* self,
                          ErrorCallback ecb,
                          Ptr<const Packet> p,
                          const Ipv4Header& header,
                          Socket::SocketErrno err);

    bool AttackActiveNow() const;

    AttackType m_attack = NONE;
    double m_dropProb = 0.0;
    Time m_start = Seconds(0);
    Time m_stop = Seconds(0);
    Ptr<UniformRandomVariable> m_rng;
};

} // namespace ns3

#endif // TRUST_MANET_MALICIOUS_ROUTING_H


Overwriting ns3/malicious-routing.h


In [6]:
%%writefile ns3/malicious-routing.cc
/*
 * TRUST-MANET : malicious-routing.cc
 * See malicious-routing.h for the design rationale.
 */
#include "malicious-routing.h"

#include "ns3/log.h"
#include "ns3/simulator.h"

namespace ns3
{

NS_LOG_COMPONENT_DEFINE("MaliciousRouting");
NS_OBJECT_ENSURE_REGISTERED(MaliciousRouting);

TypeId
MaliciousRouting::GetTypeId()
{
    static TypeId tid = TypeId("ns3::MaliciousRouting")
                            .SetParent<aodv::RoutingProtocol>()
                            .SetGroupName("Internet")
                            .AddConstructor<MaliciousRouting>();
    return tid;
}

MaliciousRouting::MaliciousRouting()
{
    // UniformRandomVariable already defaults to the [0, 1) range we need,
    // so there is no need to set Min/Max (which would require ns3/double.h).
    m_rng = CreateObject<UniformRandomVariable>();
}

MaliciousRouting::~MaliciousRouting()
{
}

void
MaliciousRouting::SetAttack(AttackType type, double dropProb, Time start, Time stop)
{
    m_attack = type;
    m_dropProb = dropProb;
    m_start = start;
    m_stop = stop;
}

bool
MaliciousRouting::AttackActiveNow() const
{
    if (m_attack == NONE)
    {
        return false;
    }
    Time now = Simulator::Now();
    return (now >= m_start) && (now < m_stop);
}

void
MaliciousRouting::ForwardHook(MaliciousRouting* self,
                              UnicastForwardCallback ucb,
                              Ptr<Ipv4Route> route,
                              Ptr<const Packet> p,
                              const Ipv4Header& header)
{
    // Only *transit* (forwarded) packets reach a UnicastForwardCallback;
    // locally-delivered packets go through the LocalDeliverCallback and are
    // never affected here.
    self->m_recvForward++;

    // Protocol 17 == UDP. Application data in this framework is UDP CBR;
    // AODV control is sent over AODV's own sockets and is not forwarded via
    // this transit path, so "transit UDP" == application data.
    const bool isData = (header.GetProtocol() == 17);

    if (isData && self->AttackActiveNow())
    {
        bool drop = false;
        if (self->m_attack == BLACKHOLE)
        {
            drop = true;
        }
        else if (self->m_attack == GRAYHOLE)
        {
            drop = (self->m_rng->GetValue() < self->m_dropProb);
        }

        if (drop)
        {
            self->m_fwdDropAttack++;
            // Silently drop: do NOT invoke ucb -> packet is not forwarded.
            return;
        }
    }

    self->m_fwdOk++;
    ucb(route, p, header);
}

void
MaliciousRouting::ErrorHook(MaliciousRouting* self,
                            ErrorCallback ecb,
                            Ptr<const Packet> p,
                            const Ipv4Header& header,
                            Socket::SocketErrno err)
{
    // Invoked by AODV when a transit packet cannot be routed (no route /
    // broken route). Counted for every node (benign + malicious) so the
    // feature pipeline can separate "malicious drop" from "no-route drop".
    self->m_noRoute++;
    ecb(p, header, err);
}

bool
MaliciousRouting::RouteInput(Ptr<const Packet> p,
                             const Ipv4Header& header,
                             Ptr<const NetDevice> idev,
                             const UnicastForwardCallback& ucb,
                             const MulticastForwardCallback& mcb,
                             const LocalDeliverCallback& lcb,
                             const ErrorCallback& ecb)
{
    // Wrap both the unicast-forward and error callbacks so our counters fire
    // whether forwarding happens synchronously or after AODV finishes a
    // route discovery. Local delivery (lcb) and multicast (mcb) are passed
    // through untouched.
    UnicastForwardCallback wrappedUcb = MakeBoundCallback(&MaliciousRouting::ForwardHook, this, ucb);
    ErrorCallback wrappedEcb = MakeBoundCallback(&MaliciousRouting::ErrorHook, this, ecb);

    return aodv::RoutingProtocol::RouteInput(p, header, idev, wrappedUcb, mcb, lcb, wrappedEcb);
}

} // namespace ns3


Writing ns3/malicious-routing.cc


In [7]:
%%writefile ns3/trust-manet-sim.cc
/*
 * TRUST-MANET : trust-manet-sim.cc
 * --------------------------------
 * One parameterised ns-3 run of an AODV MANET under a chosen attack mode.
 * Produces three files per run:
 *
 *   <prefix>_nodes.csv : per-window, per-node measurements (deltas within the window)
 *   <prefix>_edges.csv : per-window 1-hop adjacency (node pairs within rangeM)
 *   <prefix>_meta.json : the run configuration + ground-truth malicious node IDs
 *
 * The Python pipeline (scripts/build_dataset.py) turns these into the
 * per-(observer, neighbour, window) Parquet dataset described in the proposal.
 *
 * Install: copy this file + malicious-routing.{h,cc} into
 *          <ns3>/scratch/trust-manet/  and build with
 *          ./ns3 build trust-manet
 *
 * Targeted ns-3 version: ns-3.42.
 */
#include "malicious-routing.h"

#include "ns3/applications-module.h"
#include "ns3/core-module.h"
#include "ns3/internet-module.h"
#include "ns3/mobility-module.h"
#include "ns3/network-module.h"
#include "ns3/wifi-module.h"

#include <algorithm>
#include <cmath>
#include <fstream>
#include <numeric>
#include <random>
#include <sstream>
#include <string>
#include <vector>

using namespace ns3;

NS_LOG_COMPONENT_DEFINE("TrustManetSim");

// ---------------------------------------------------------------------------
// Routing helper that installs MaliciousRouting (an AODV subclass) on every
// node. Maliciousness is switched on afterwards per selected node.
// ---------------------------------------------------------------------------
class MaliciousAodvHelper : public Ipv4RoutingHelper
{
  public:
    MaliciousAodvHelper() { m_factory.SetTypeId("ns3::MaliciousRouting"); }

    MaliciousAodvHelper* Copy() const override { return new MaliciousAodvHelper(*this); }

    Ptr<Ipv4RoutingProtocol> Create(Ptr<Node> node) const override
    {
        Ptr<MaliciousRouting> agent = m_factory.Create<MaliciousRouting>();
        node->AggregateObject(agent);
        return agent;
    }

  private:
    ObjectFactory m_factory;
};

// ---------------------------------------------------------------------------
// Application traffic counters (indexed by node id).
// ---------------------------------------------------------------------------
static void
TxSink(std::vector<uint64_t>* v, uint32_t node, Ptr<const Packet>)
{
    (*v)[node]++;
}

static void
RxSink(std::vector<uint64_t>* v, uint32_t node, Ptr<const Packet>, const Address&)
{
    (*v)[node]++;
}

// ---------------------------------------------------------------------------
// Per-window collector.
// ---------------------------------------------------------------------------
struct Collector
{
    NodeContainer nodes;
    uint32_t nNodes = 0;
    double windowSec = 5.0;
    double rangeM = 120.0;
    double simStop = 200.0;

    std::ofstream nodesOut;
    std::ofstream edgesOut;

    std::vector<uint64_t> pRecv, pOk, pDrop, pNoRoute, pTx, pRx; // previous cumulative
    std::vector<uint64_t>* appTx = nullptr;
    std::vector<uint64_t>* appRx = nullptr;

    uint32_t widx = 0;

    void Start()
    {
        pRecv.assign(nNodes, 0);
        pOk.assign(nNodes, 0);
        pDrop.assign(nNodes, 0);
        pNoRoute.assign(nNodes, 0);
        pTx.assign(nNodes, 0);
        pRx.assign(nNodes, 0);

        nodesOut << "window_idx,time_s,node,pos_x,pos_y,vel_x,vel_y,speed,"
                    "recv_forward,fwd_ok,fwd_drop_attack,no_route,app_tx,app_rx\n";
        edgesOut << "window_idx,i,j,dist\n";

        Simulator::Schedule(Seconds(windowSec), &Collector::Sample, this);
    }

    void Sample()
    {
        const double t = Simulator::Now().GetSeconds();
        std::vector<Vector> pos(nNodes);

        for (uint32_t n = 0; n < nNodes; ++n)
        {
            Ptr<Node> node = nodes.Get(n);
            Ptr<MobilityModel> mm = node->GetObject<MobilityModel>();
            Vector p = mm->GetPosition();
            Vector v = mm->GetVelocity();
            pos[n] = p;

            Ptr<MaliciousRouting> r = node->GetObject<MaliciousRouting>();

            const uint64_t recv = r->m_recvForward;
            const uint64_t ok = r->m_fwdOk;
            const uint64_t drop = r->m_fwdDropAttack;
            const uint64_t nr = r->m_noRoute;
            const uint64_t tx = (*appTx)[n];
            const uint64_t rx = (*appRx)[n];

            const uint64_t dRecv = recv - pRecv[n];
            const uint64_t dOk = ok - pOk[n];
            const uint64_t dDrop = drop - pDrop[n];
            const uint64_t dNr = nr - pNoRoute[n];
            const uint64_t dTx = tx - pTx[n];
            const uint64_t dRx = rx - pRx[n];

            const double speed = std::sqrt(v.x * v.x + v.y * v.y + v.z * v.z);

            nodesOut << widx << ',' << t << ',' << n << ',' << p.x << ',' << p.y << ',' << v.x
                     << ',' << v.y << ',' << speed << ',' << dRecv << ',' << dOk << ',' << dDrop
                     << ',' << dNr << ',' << dTx << ',' << dRx << '\n';

            pRecv[n] = recv;
            pOk[n] = ok;
            pDrop[n] = drop;
            pNoRoute[n] = nr;
            pTx[n] = tx;
            pRx[n] = rx;
        }

        // 1-hop adjacency within observation range.
        for (uint32_t i = 0; i < nNodes; ++i)
        {
            for (uint32_t j = i + 1; j < nNodes; ++j)
            {
                const double d = CalculateDistance(pos[i], pos[j]);
                if (d <= rangeM)
                {
                    edgesOut << widx << ',' << i << ',' << j << ',' << d << '\n';
                }
            }
        }

        widx++;
        if (Simulator::Now().GetSeconds() + windowSec <= simStop + 1e-9)
        {
            Simulator::Schedule(Seconds(windowSec), &Collector::Sample, this);
        }
    }
};

// ---------------------------------------------------------------------------
int
main(int argc, char* argv[])
{
    // ---- parameters ----
    uint32_t nNodes = 50;
    double speed = 5.0;             // m/s (0 => static)
    double compromisedFrac = 0.10;  // fraction of nodes that are attackers
    std::string attackType = "blackhole"; // clean | blackhole | grayhole
    double grayholeDropProb = 0.5;
    uint32_t nFlows = 15;
    double simTime = 200.0;
    double windowSec = 5.0;
    double attackStart = 20.0;
    double appStart = 2.0;
    double areaX = 1500.0;
    double areaY = 1500.0;
    double rangeM = 120.0;
    uint32_t seed = 1;
    std::string outPrefix = "run";

    CommandLine cmd(__FILE__);
    cmd.AddValue("nNodes", "Number of nodes", nNodes);
    cmd.AddValue("speed", "Max node speed in m/s (0 = static)", speed);
    cmd.AddValue("compromisedFrac", "Fraction of compromised nodes", compromisedFrac);
    cmd.AddValue("attackType", "clean | blackhole | grayhole", attackType);
    cmd.AddValue("grayholeDropProb", "Drop probability for grayhole", grayholeDropProb);
    cmd.AddValue("nFlows", "Number of CBR UDP flows", nFlows);
    cmd.AddValue("simTime", "Simulation time (s)", simTime);
    cmd.AddValue("windowSec", "Measurement window (s)", windowSec);
    cmd.AddValue("attackStart", "Attack start time (s)", attackStart);
    cmd.AddValue("appStart", "Application start time (s)", appStart);
    cmd.AddValue("areaX", "Area width (m)", areaX);
    cmd.AddValue("areaY", "Area height (m)", areaY);
    cmd.AddValue("rangeM", "Observation/adjacency range (m)", rangeM);
    cmd.AddValue("seed", "RNG run (also seeds node/flow selection)", seed);
    cmd.AddValue("outPrefix", "Output file prefix", outPrefix);
    cmd.Parse(argc, argv);

    SeedManager::SetSeed(1);
    SeedManager::SetRun(seed);

    // Deterministic (seed-controlled) selection independent of ns-3 streams.
    std::mt19937 gen(static_cast<uint32_t>(seed) * 2654435761u + 12345u);

    NodeContainer nodes;
    nodes.Create(nNodes);

    // ---- Wi-Fi (802.11b ad hoc) ----
    WifiHelper wifi;
    wifi.SetStandard(WIFI_STANDARD_80211b);
    wifi.SetRemoteStationManager("ns3::ConstantRateWifiManager",
                                 "DataMode",
                                 StringValue("DsssRate11Mbps"),
                                 "ControlMode",
                                 StringValue("DsssRate1Mbps"));

    YansWifiChannelHelper channel = YansWifiChannelHelper::Default();
    YansWifiPhyHelper phy;
    phy.SetChannel(channel.Create());

    WifiMacHelper mac;
    mac.SetType("ns3::AdhocWifiMac");

    NetDeviceContainer devices = wifi.Install(phy, mac, nodes);

    // ---- mobility ----
    MobilityHelper mob;
    Ptr<RandomRectanglePositionAllocator> posAlloc =
        CreateObject<RandomRectanglePositionAllocator>();
    {
        std::ostringstream sx, sy;
        sx << "ns3::UniformRandomVariable[Min=0|Max=" << areaX << "]";
        sy << "ns3::UniformRandomVariable[Min=0|Max=" << areaY << "]";
        posAlloc->SetAttribute("X", StringValue(sx.str()));
        posAlloc->SetAttribute("Y", StringValue(sy.str()));
    }
    mob.SetPositionAllocator(posAlloc);

    if (speed <= 0.0)
    {
        mob.SetMobilityModel("ns3::ConstantPositionMobilityModel");
    }
    else
    {
        std::ostringstream sp;
        sp << "ns3::ConstantRandomVariable[Constant=" << speed << "]";
        mob.SetMobilityModel("ns3::RandomWaypointMobilityModel",
                             "Speed",
                             StringValue(sp.str()),
                             "Pause",
                             StringValue("ns3::ConstantRandomVariable[Constant=0.0]"),
                             "PositionAllocator",
                             PointerValue(posAlloc));
    }
    mob.Install(nodes);

    // ---- internet stack with malicious-capable AODV ----
    MaliciousAodvHelper malHelper;
    InternetStackHelper internet;
    internet.SetRoutingHelper(malHelper);
    internet.Install(nodes);

    Ipv4AddressHelper address;
    address.SetBase("10.1.0.0", "255.255.0.0"); // /16 supports up to ~65k hosts
    Ipv4InterfaceContainer ifaces = address.Assign(devices);

    // ---- choose malicious nodes ----
    std::vector<uint32_t> idx(nNodes);
    std::iota(idx.begin(), idx.end(), 0);
    std::shuffle(idx.begin(), idx.end(), gen);

    uint32_t nMal = 0;
    MaliciousRouting::AttackType atk = MaliciousRouting::NONE;
    if (attackType == "blackhole")
    {
        atk = MaliciousRouting::BLACKHOLE;
    }
    else if (attackType == "grayhole")
    {
        atk = MaliciousRouting::GRAYHOLE;
    }
    else
    {
        atk = MaliciousRouting::NONE; // "clean"
    }

    if (atk != MaliciousRouting::NONE)
    {
        nMal = static_cast<uint32_t>(std::floor(compromisedFrac * nNodes));
    }

    std::vector<uint32_t> malicious(idx.begin(), idx.begin() + nMal);
    std::sort(malicious.begin(), malicious.end());

    for (uint32_t m : malicious)
    {
        Ptr<MaliciousRouting> r = nodes.Get(m)->GetObject<MaliciousRouting>();
        r->SetAttack(atk, grayholeDropProb, Seconds(attackStart), Seconds(simTime));
    }

    // ---- traffic: nFlows random source->destination CBR UDP flows ----
    std::vector<uint64_t> appTx(nNodes, 0), appRx(nNodes, 0);
    std::uniform_int_distribution<uint32_t> pick(0, nNodes - 1);

    for (uint32_t f = 0; f < nFlows; ++f)
    {
        uint32_t src = pick(gen);
        uint32_t dst = pick(gen);
        while (dst == src)
        {
            dst = pick(gen);
        }
        uint16_t port = static_cast<uint16_t>(9000 + f);

        OnOffHelper onoff("ns3::UdpSocketFactory",
                          InetSocketAddress(ifaces.GetAddress(dst), port));
        onoff.SetAttribute("DataRate", StringValue("16kbps"));
        onoff.SetAttribute("PacketSize", UintegerValue(512));
        onoff.SetAttribute("OnTime", StringValue("ns3::ConstantRandomVariable[Constant=1]"));
        onoff.SetAttribute("OffTime", StringValue("ns3::ConstantRandomVariable[Constant=0]"));
        ApplicationContainer srcApp = onoff.Install(nodes.Get(src));
        srcApp.Start(Seconds(appStart));
        srcApp.Stop(Seconds(simTime - 1.0));

        PacketSinkHelper sink("ns3::UdpSocketFactory",
                              InetSocketAddress(Ipv4Address::GetAny(), port));
        ApplicationContainer sinkApp = sink.Install(nodes.Get(dst));
        sinkApp.Start(Seconds(1.0));
        sinkApp.Stop(Seconds(simTime));

        srcApp.Get(0)->TraceConnectWithoutContext("Tx", MakeBoundCallback(&TxSink, &appTx, src));
        sinkApp.Get(0)->TraceConnectWithoutContext("Rx", MakeBoundCallback(&RxSink, &appRx, dst));
    }

    // ---- collector ----
    Collector col;
    col.nodes = nodes;
    col.nNodes = nNodes;
    col.windowSec = windowSec;
    col.rangeM = rangeM;
    col.simStop = simTime;
    col.appTx = &appTx;
    col.appRx = &appRx;
    col.nodesOut.open(outPrefix + "_nodes.csv");
    col.edgesOut.open(outPrefix + "_edges.csv");
    col.Start();

    // ---- meta ----
    {
        std::ofstream meta(outPrefix + "_meta.json");
        meta << "{\n";
        meta << "  \"n_nodes\": " << nNodes << ",\n";
        meta << "  \"speed\": " << speed << ",\n";
        meta << "  \"compromised_frac\": " << compromisedFrac << ",\n";
        meta << "  \"attack_type\": \"" << attackType << "\",\n";
        meta << "  \"grayhole_drop_prob\": " << grayholeDropProb << ",\n";
        meta << "  \"n_flows\": " << nFlows << ",\n";
        meta << "  \"sim_time\": " << simTime << ",\n";
        meta << "  \"window_sec\": " << windowSec << ",\n";
        meta << "  \"attack_start\": " << attackStart << ",\n";
        meta << "  \"app_start\": " << appStart << ",\n";
        meta << "  \"area_x\": " << areaX << ",\n";
        meta << "  \"area_y\": " << areaY << ",\n";
        meta << "  \"range_m\": " << rangeM << ",\n";
        meta << "  \"seed\": " << seed << ",\n";
        meta << "  \"malicious_nodes\": [";
        for (size_t k = 0; k < malicious.size(); ++k)
        {
            meta << malicious[k] << (k + 1 < malicious.size() ? ", " : "");
        }
        meta << "]\n}\n";
    }

    Simulator::Stop(Seconds(simTime));
    Simulator::Run();
    Simulator::Destroy();

    col.nodesOut.close();
    col.edgesOut.close();
    return 0;
}


Writing ns3/trust-manet-sim.cc


In [8]:
%%writefile scripts/build_ns3.sh
#!/usr/bin/env bash
# ---------------------------------------------------------------------------
# build_ns3.sh : fetch a pinned ns-3, install the TRUST-MANET scratch program,
#                and build only the modules we need (fast even on 1 core).
#
# Usage:
#   bash scripts/build_ns3.sh [NS3_DIR] [NS3_VERSION]
#
#   NS3_DIR      where to clone ns-3   (default: ./ns-3-dev)
#   NS3_VERSION  git tag to check out  (default: ns-3.42)
#
# After this finishes, run the matrix with scripts/run_matrix.py.
# ---------------------------------------------------------------------------
set -euo pipefail

NS3_DIR="${1:-$(pwd)/ns-3-dev}"
NS3_VERSION="${2:-ns-3.42}"
REPO="https://github.com/nsnam/ns-3-dev-git.git"

HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"   # project root

echo "==> Checking build prerequisites"
missing=()
for t in git g++ cmake python3 make; do
  command -v "$t" >/dev/null 2>&1 || missing+=("$t")
done
if [ "${#missing[@]}" -gt 0 ]; then
  echo "ERROR: missing tools: ${missing[*]}"
  echo "On Ubuntu/Debian:  sudo apt-get update && sudo apt-get install -y git g++ cmake python3 make"
  exit 1
fi

# Clone when there's no real git checkout yet. Checking for .git (not just the
# directory) means a pre-created empty dir or a half-finished clone won't block it.
if [ ! -d "$NS3_DIR/.git" ]; then
  echo "==> Cloning ns-3 ($NS3_VERSION) into $NS3_DIR"
  rm -rf "$NS3_DIR"
  git clone "$REPO" "$NS3_DIR"
fi

cd "$NS3_DIR"
echo "==> Checking out $NS3_VERSION"
git fetch --tags --quiet || true
git checkout "$NS3_VERSION"

echo "==> Installing TRUST-MANET scratch program"
mkdir -p scratch/trust-manet
# Name the main file after the directory so the run target is simply 'trust-manet'.
cp "$HERE/ns3/trust-manet-sim.cc"   scratch/trust-manet/trust-manet.cc
cp "$HERE/ns3/malicious-routing.cc" scratch/trust-manet/
cp "$HERE/ns3/malicious-routing.h"  scratch/trust-manet/
rm -f scratch/trust-manet/trust-manet-sim.cc   # drop any stale duplicate main

echo "==> Configuring (optimized, minimal module set)"
# NOTE: ns-3 expects a SEMICOLON-separated, quoted module list; dependencies are
# resolved automatically. Limiting modules cuts build time/memory. Add modules
# here if you extend the simulation (e.g. flow-monitor, energy).
./ns3 clean || true
./ns3 configure -d optimized \
  --enable-modules='core;network;internet;wifi;mobility;aodv;applications;propagation' \
  --disable-examples --disable-tests

echo "==> Building"
# Builds the enabled libraries + the scratch program. (Building by explicit
# target name is avoided because the short name is not accepted on every version.)
./ns3 build

echo
echo "==> Done. ns-3 is at: $NS3_DIR"
echo "    Smoke test:"
echo "    (cd \"$NS3_DIR\" && ./ns3 run \"trust-manet --nNodes=25 --attackType=blackhole --simTime=60 --outPrefix=/tmp/smoke\")"


Writing scripts/build_ns3.sh


In [10]:
%%writefile scripts/run_matrix.py
#!/usr/bin/env python3
"""
run_matrix.py
-------------
Enumerate the TRUST-MANET scenario matrix and run the compiled ns-3 program
once per (sim-factor combination x seed). Raw outputs (<run_id>_nodes.csv,
<run_id>_edges.csv, <run_id>_meta.json) are written into --out.

Prereq: build the simulator first with scripts/build_ns3.sh.

Examples
--------
# fast screening pass (36 runs) to validate the pipeline:
python scripts/run_matrix.py --ns3-dir ./ns-3-dev --which screening --out raw/

# full matrix (450 runs) -- long; use --resume to restart safely:
python scripts/run_matrix.py --ns3-dir ./ns-3-dev --which full --out raw/ --resume
"""
import argparse
import math
import os
import subprocess
import sys
import csv

try:
    import yaml
except ImportError:
    sys.exit("PyYAML missing. Run: pip install -r requirements.txt")


def enumerate_runs(cfg, which):
    m = cfg[which]
    fixed = cfg["fixed"]
    runs = []
    for n in m["sim_factors"]["n_nodes"]:
        for speed in m["sim_factors"]["speed"]:
            for atk in m["sim_factors"]["attack"]:
                for seed in m["seeds"]:
                    attack_type = atk["attack_type"]
                    cfrac = float(atk["compromised_frac"])
                    scenario_id = f"n{n}_s{speed}_{attack_type}_c{cfrac:.2f}"
                    run_id = f"{scenario_id}_seed{seed}"
                    # derived
                    area = n * float(fixed["density_m2_per_node"])
                    side = round(math.sqrt(area), 1)
                    n_flows = max(1, round(fixed["n_flows_per_25_nodes"] * n / 25.0))
                    runs.append(
                        dict(
                            run_id=run_id,
                            scenario_id=scenario_id,
                            n_nodes=n,
                            speed=speed,
                            attack_type=attack_type,
                            compromised_frac=cfrac,
                            seed=seed,
                            area_x=side,
                            area_y=side,
                            n_flows=n_flows,
                        )
                    )
    return runs, fixed


def build_cmd(run, fixed, out_dir):
    prefix = os.path.join(os.path.abspath(out_dir), run["run_id"])
    args = [
        f"--nNodes={run['n_nodes']}",
        f"--speed={run['speed']}",
        f"--attackType={run['attack_type']}",
        f"--compromisedFrac={run['compromised_frac']}",
        f"--grayholeDropProb={fixed['grayhole_drop_prob']}",
        f"--nFlows={run['n_flows']}",
        f"--simTime={fixed['sim_time']}",
        f"--windowSec={fixed['window_sec']}",
        f"--attackStart={fixed['attack_start']}",
        f"--appStart={fixed['app_start']}",
        f"--areaX={run['area_x']}",
        f"--areaY={run['area_y']}",
        f"--rangeM={fixed['range_m']}",
        f"--seed={run['seed']}",
        f"--outPrefix={prefix}",
    ]
    return prefix, "trust-manet " + " ".join(args)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--ns3-dir", required=True, help="Path to the built ns-3 tree")
    ap.add_argument("--matrix", default="scenario_matrix.yaml")
    ap.add_argument("--which", default="screening", choices=["full", "screening"])
    ap.add_argument("--out", default="raw", help="Output dir for raw run files")
    ap.add_argument("--limit", type=int, default=0, help="Run only the first N (0 = all)")
    ap.add_argument("--resume", action="store_true", help="Skip runs whose _meta.json exists")
    ap.add_argument("--dry-run", action="store_true", help="Print commands, do not execute")
    args = ap.parse_args()

    with open(args.matrix) as fh:
        cfg = yaml.safe_load(fh)

    os.makedirs(args.out, exist_ok=True)
    runs, fixed = enumerate_runs(cfg, args.which)
    if args.limit:
        runs = runs[: args.limit]

    # write the run index
    idx_path = os.path.join(args.out, "run_index.csv")
    with open(idx_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(runs[0].keys()))
        w.writeheader()
        w.writerows(runs)
    print(f"{len(runs)} runs enumerated -> {idx_path}")

    ns3 = os.path.abspath(args.ns3_dir)
    ok, skipped, failed = 0, 0, 0
    for i, run in enumerate(runs, 1):
        prefix, program = build_cmd(run, fixed, args.out)
        meta = prefix + "_meta.json"
        if args.resume and os.path.exists(meta):
            skipped += 1
            continue
        cmd = ["./ns3", "run", "--no-build", program]
        print(f"[{i}/{len(runs)}] {run['run_id']}")
        if args.dry_run:
            print("   ", " ".join(cmd))
            continue
        r = subprocess.run(cmd, cwd=ns3)
        if r.returncode == 0 and os.path.exists(meta):
            ok += 1
        else:
            failed += 1
            print(f"   !! run failed (rc={r.returncode})")

    print(f"\nDone. ok={ok} skipped={skipped} failed={failed}")
    if failed:
        sys.exit(1)


if __name__ == "__main__":
    main()


Overwriting scripts/run_matrix.py


In [11]:
%%writefile scripts/build_dataset.py
#!/usr/bin/env python3
"""
build_dataset.py
----------------
Turn raw ns-3 run logs (from run_matrix.py) into the TRUST-MANET dataset:
one row per (observer i, observed neighbour j, time window t), with the five
feature groups from the proposal plus targets, written to Parquet.

For each run we:
  1. load per-window node measurements and 1-hop adjacency,
  2. build directed observer->neighbour pairs (i watches j),
  3. compute mobility / link / forwarding features,
  4. simulate honest + corrupt observer reports (bad-mouthing / ballot-stuffing)
     for every corrupt_reporter_frac in the matrix (NO re-simulation),
  5. aggregate other observers' reports about j,
  6. assign targets, and write one Parquet file per run into --out.

Only simulator-provided ground truth is used for LABELS; the watchdog
forward-ratio the observer "sees" is noised and is a FEATURE. Link RSSI is a
distance-based proxy and is named rssi_proxy_dbm accordingly (see the proposal:
mark approximations instead of inventing sensor access).

Usage:
  python scripts/build_dataset.py --raw raw/ --out dataset/ --matrix scenario_matrix.yaml
"""
import argparse
import glob
import json
import os
import sys
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    sys.exit("PyYAML missing. Run: pip install -r requirements.txt")

# ---- distance -> RSSI proxy (log-distance path loss; documented approximation)
PT_DBM = 16.0206   # ns-3 default Wi-Fi tx power
PL0_DB = 40.0      # reference path loss at d0
PL_EXP = 3.0       # path-loss exponent
D0 = 1.0
RSSI_MIN, RSSI_MAX = -95.0, -40.0


def rssi_proxy(dist):
    d = np.maximum(dist, D0)
    pl = PL0_DB + 10.0 * PL_EXP * np.log10(d / D0)
    return PT_DBM - pl


def link_quality(rssi):
    return np.clip((rssi - RSSI_MIN) / (RSSI_MAX - RSSI_MIN), 0.0, 1.0)


def neighbour_sets(edges):
    """window_idx -> {node: set(neighbours)} (undirected)."""
    adj = defaultdict(lambda: defaultdict(set))
    for w, i, j in edges[["window_idx", "i", "j"]].itertuples(index=False):
        adj[w][i].add(j)
        adj[w][j].add(i)
    return adj


def neighbour_churn(adj, windows):
    """(window_idx, node) -> churn = 1 - Jaccard(N_t, N_{t-1})."""
    churn = {}
    wsorted = sorted(windows)
    allnodes = set()
    for w in wsorted:
        allnodes |= set(adj[w].keys())
    for node in allnodes:
        prev = None
        for w in wsorted:
            cur = adj[w].get(node, set())
            if prev is None:
                churn[(w, node)] = 0.0
            else:
                union = prev | cur
                inter = prev & cur
                churn[(w, node)] = 0.0 if not union else 1.0 - len(inter) / len(union)
            prev = cur
    return churn


def process_run(meta_path, out_dir, post_fracs, rmodel):
    prefix = meta_path[: -len("_meta.json")]
    run_id = os.path.basename(prefix)
    with open(meta_path) as fh:
        meta = json.load(fh)

    nodes = pd.read_csv(prefix + "_nodes.csv")
    edges = pd.read_csv(prefix + "_edges.csv")
    if nodes.empty or edges.empty:
        print(f"  skip {run_id}: empty logs")
        return

    mal_set = set(meta.get("malicious_nodes", []))
    attack_type = meta["attack_type"]
    attack_start = float(meta["attack_start"])
    window_sec = float(meta["window_sec"])
    n_nodes = int(meta["n_nodes"])

    # --- per-node per-window derived quantities ------------------------------
    nodes["fwd_ratio_true"] = np.where(
        nodes["recv_forward"] > 0, nodes["fwd_ok"] / nodes["recv_forward"], np.nan
    )
    nodes["attack_active"] = (attack_type != "clean") & (nodes["time_s"] >= attack_start)
    nodes["is_mal_active"] = nodes["node"].isin(mal_set) & nodes["attack_active"]

    # window-level reachability (global PDR in the window)
    win_tot = nodes.groupby("window_idx").agg(
        app_tx_sum=("app_tx", "sum"), app_rx_sum=("app_rx", "sum")
    )
    win_tot["route_reachability"] = np.where(
        win_tot["app_tx_sum"] > 0, win_tot["app_rx_sum"] / win_tot["app_tx_sum"], np.nan
    ).clip(0, 1)

    windows = nodes["window_idx"].unique()
    adj = neighbour_sets(edges)
    churn = neighbour_churn(adj, windows)

    # --- directed observer->neighbour pairs ----------------------------------
    d1 = edges.rename(columns={"i": "obs", "j": "tgt"})
    d2 = edges.rename(columns={"i": "tgt", "j": "obs"})
    pairs = pd.concat([d1, d2], ignore_index=True)[["window_idx", "obs", "tgt", "dist"]]

    nf = nodes[
        ["window_idx", "node", "time_s", "vel_x", "vel_y", "speed",
         "fwd_ratio_true", "recv_forward", "fwd_drop_attack", "no_route",
         "is_mal_active"]
    ]
    pairs = pairs.merge(
        nf.add_suffix("_obs").rename(columns={"window_idx_obs": "window_idx", "node_obs": "obs"}),
        on=["window_idx", "obs"], how="left",
    )
    pairs = pairs.merge(
        nf.add_suffix("_tgt").rename(columns={"window_idx_tgt": "window_idx", "node_tgt": "tgt"}),
        on=["window_idx", "tgt"], how="left",
    )

    # ---- mobility features ----
    pairs["rel_speed"] = np.hypot(pairs["vel_x_obs"] - pairs["vel_x_tgt"],
                                  pairs["vel_y_obs"] - pairs["vel_y_tgt"])
    pairs["neighbor_churn"] = [churn.get((w, o), 0.0)
                               for w, o in zip(pairs["window_idx"], pairs["obs"])]

    # contact lifetime (consecutive windows the pair stays adjacent)
    pairs = pairs.sort_values(["obs", "tgt", "window_idx"]).reset_index(drop=True)
    wdiff = pairs.groupby(["obs", "tgt"])["window_idx"].diff()
    isnew = wdiff.ne(1.0)  # NaN (first in group) -> True
    runid = isnew.groupby([pairs["obs"], pairs["tgt"]]).cumsum()
    pairs["contact_lifetime_win"] = pairs.groupby(["obs", "tgt", runid]).cumcount() + 1
    pairs["contact_lifetime_s"] = pairs["contact_lifetime_win"] * window_sec

    # ---- link features (distance-based proxies) ----
    pairs["rssi_proxy_dbm"] = rssi_proxy(pairs["dist"].to_numpy())
    pairs["link_quality_proxy"] = link_quality(pairs["rssi_proxy_dbm"].to_numpy())

    # ---- forwarding features (watchdog view of the TARGET's behaviour) ----
    rng = np.random.default_rng(abs(hash(run_id)) % (2**32))
    base_ratio = pairs["fwd_ratio_true_tgt"].to_numpy()
    noise = rng.normal(0.0, rmodel["observation_sigma"], size=len(pairs))
    observed = np.clip(base_ratio + noise, 0.0, 1.0)
    enough = pairs["recv_forward_tgt"].to_numpy() >= rmodel["min_transit_for_obs"]
    observed = np.where(enough & ~np.isnan(base_ratio), observed, np.nan)
    pairs["observed_fwd_ratio"] = observed
    pairs["obs_confidence"] = np.where(enough & ~np.isnan(base_ratio),
                                       np.clip(pairs["recv_forward_tgt"] / 20.0, 0, 1), 0.0)
    pairs["missing_ack_ratio"] = np.where(np.isnan(observed), np.nan, 1.0 - observed)
    pairs["tgt_transit_load"] = pairs["recv_forward_tgt"]
    pairs["tgt_no_route"] = pairs["no_route_tgt"]

    # ---- targets ----
    pairs["tgt_malicious"] = pairs["is_mal_active_tgt"].fillna(False).astype(int)
    pairs["attack_type"] = attack_type
    pairs["tgt_drop_attack_true"] = pairs["fwd_drop_attack_tgt"].fillna(0)
    pairs = pairs.merge(win_tot[["route_reachability"]], on="window_idx", how="left")

    # scenario / key metadata
    pairs["run_id"] = run_id
    pairs["scenario_id"] = f"n{n_nodes}_s{meta['speed']}_{attack_type}_c{meta['compromised_frac']:.2f}"
    pairs["seed"] = meta["seed"]
    pairs["n_nodes"] = n_nodes
    pairs["speed"] = meta["speed"]
    pairs["compromised_frac"] = meta["compromised_frac"]
    pairs["time_s"] = pairs["time_s_obs"]

    frames = []
    for frac in post_fracs:
        frames.append(apply_reports(pairs.copy(), frac, n_nodes, meta["seed"], rmodel))
    out = pd.concat(frames, ignore_index=True)

    # drop intermediate suffix columns the consumer doesn't need
    drop = [c for c in out.columns if c.endswith("_obs") or c.endswith("_tgt")]
    keep_tgt = {"tgt"}  # keep the id
    drop = [c for c in drop if c not in keep_tgt]
    out = out.drop(columns=[c for c in drop if c in out.columns], errors="ignore")
    out = out.rename(columns={"obs": "observer_id", "tgt": "neighbor_id"})

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"part-{run_id}.parquet")
    out.to_parquet(out_path, index=False)
    print(f"  wrote {out_path}  rows={len(out)}")


def apply_reports(df, corrupt_frac, n_nodes, seed, rmodel):
    """Generate honest + corrupt observer reports for one corrupt_reporter_frac."""
    rng = np.random.default_rng((int(seed) * 1000003 + int(corrupt_frac * 1000)) % (2**32))

    n_corrupt = int(np.floor(corrupt_frac * n_nodes))
    corrupt_nodes = set(rng.choice(np.arange(n_nodes), size=n_corrupt, replace=False).tolist()) \
        if n_corrupt > 0 else set()

    obs = df["obs"].to_numpy()
    tgt_mal = df["tgt_malicious"].to_numpy().astype(bool)
    observed = df["observed_fwd_ratio"].to_numpy()

    # honest risk: high when the target appears to drop packets
    honest = np.where(np.isnan(observed), np.nan, np.clip(1.0 - observed, 0.0, 1.0))

    obs_corrupt = np.array([o in corrupt_nodes for o in obs])
    badmouth = rng.uniform(rmodel["badmouth_risk_min"], 1.0, size=len(df))
    ballot = rng.uniform(0.0, rmodel["ballotstuff_risk_max"], size=len(df))

    risk = honest.copy()
    # corrupt reporter bad-mouths benign targets (inflate risk)
    m_bad = obs_corrupt & ~tgt_mal
    risk = np.where(m_bad, np.maximum(np.nan_to_num(honest, nan=0.0), badmouth), risk)
    # corrupt reporter ballot-stuffs for malicious (colluding) targets (deflate)
    m_ball = obs_corrupt & tgt_mal
    risk = np.where(m_ball, np.minimum(np.nan_to_num(honest, nan=1.0), ballot), risk)

    df["corrupt_reporter_frac"] = corrupt_frac
    df["observer_is_corrupt"] = obs_corrupt.astype(int)
    df["reported_risk"] = risk
    # report freshness within the window (synthetic; models staleness jitter)
    df["report_age_s"] = rng.uniform(0.0, 5.0, size=len(df))
    df["has_report"] = (~np.isnan(risk)).astype(int)

    # ---- causal observer consistency: running agreement with ground truth ----
    df["agree"] = np.where(
        np.isnan(risk), np.nan, ((risk > 0.5).astype(float) == tgt_mal.astype(float)).astype(float)
    )
    df = df.sort_values(["obs", "window_idx"]).reset_index(drop=True)
    # mean agreement per (obs, window), then expanding mean over strictly-earlier windows
    ow = df.groupby(["obs", "window_idx"])["agree"].mean().reset_index(name="win_agree")
    ow = ow.sort_values(["obs", "window_idx"])
    ow["cum"] = ow.groupby("obs")["win_agree"].transform(
        lambda s: s.shift(1).expanding().mean()
    )
    ow["observer_consistency"] = ow["cum"].fillna(0.5)
    df = df.merge(ow[["obs", "window_idx", "observer_consistency"]],
                  on=["obs", "window_idx"], how="left")

    # ---- aggregate OTHER observers' reports about the target ----
    rep = df[df["has_report"] == 1]
    grp = rep.groupby(["window_idx", "tgt"]).agg(
        sum_risk=("reported_risk", "sum"),
        cnt=("reported_risk", "size"),
        sum_age=("report_age_s", "sum"),
        sum_cons=("observer_consistency", "sum"),
    ).reset_index()
    df = df.merge(grp, on=["window_idx", "tgt"], how="left")

    own_risk = np.where(df["has_report"] == 1, df["reported_risk"], 0.0)
    own_age = np.where(df["has_report"] == 1, df["report_age_s"], 0.0)
    own_cons = np.where(df["has_report"] == 1, df["observer_consistency"], 0.0)
    own_in = (df["has_report"] == 1).astype(float)

    others_cnt = (df["cnt"].fillna(0) - own_in).clip(lower=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        df["rep_count_others"] = others_cnt
        df["mean_reported_risk_others"] = np.where(
            others_cnt > 0, (df["sum_risk"].fillna(0) - own_risk) / others_cnt, np.nan)
        df["mean_report_age_others"] = np.where(
            others_cnt > 0, (df["sum_age"].fillna(0) - own_age) / others_cnt, np.nan)
        df["mean_observer_consistency_others"] = np.where(
            others_cnt > 0, (df["sum_cons"].fillna(0) - own_cons) / others_cnt, np.nan)
        # all-observer mean risk about the target (incl. self) for false-accusation label
        mean_all = np.where(df["cnt"].fillna(0) > 0, df["sum_risk"].fillna(0) / df["cnt"], np.nan)

    # false accusation: benign target that the neighbourhood collectively flags
    df["is_false_accusation"] = (
        (df["tgt_malicious"] == 0) & (pd.Series(mean_all) > 0.5)
    ).astype(int)

    df = df.drop(columns=["sum_risk", "cnt", "sum_age", "sum_cons", "agree"], errors="ignore")
    return df


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--raw", default="raw", help="Dir with raw ns-3 run files")
    ap.add_argument("--out", default="dataset", help="Output Parquet dir")
    ap.add_argument("--matrix", default="scenario_matrix.yaml")
    ap.add_argument("--which", default="screening", choices=["full", "screening"])
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    with open(args.matrix) as fh:
        cfg = yaml.safe_load(fh)
    post_fracs = cfg[args.which]["post_factors"]["corrupt_reporter_frac"]
    rmodel = cfg["report_model"]

    metas = sorted(glob.glob(os.path.join(args.raw, "*_meta.json")))
    if args.limit:
        metas = metas[: args.limit]
    if not metas:
        sys.exit(f"No *_meta.json found in {args.raw}. Run run_matrix.py first.")

    print(f"Building dataset from {len(metas)} runs x {len(post_fracs)} corrupt fracs")
    for mp in metas:
        process_run(mp, args.out, post_fracs, rmodel)

    print(f"\nDone. Parquet parts in {args.out}/  "
          f"(read with pandas.read_parquet or pyarrow.dataset)")


if __name__ == "__main__":
    main()


Writing scripts/build_dataset.py


In [12]:
%%writefile scripts/split_dataset.py
#!/usr/bin/env python3
"""
split_dataset.py
----------------
Assign a leakage-safe train/val/test split to the Parquet parts produced by
build_dataset.py. A whole simulation run (and, in 'config' mode, a whole
scenario configuration) is assigned to exactly ONE split, so no window from a
test scenario can appear in training/tuning -- as required by the proposal.

Modes:
  run     : split at the (scenario, seed) run level (default).
  config  : split at the scenario-config level -> test contains UNSEEN
            configurations (strictest cross-scenario generalization test).

Each input part is rewritten to --out with an added 'split' column.

Usage:
  python scripts/split_dataset.py --dataset dataset/ --out dataset_split/ --mode config
"""
import argparse
import glob
import os
import sys

import pandas as pd
import numpy as np


def key_of(part, mode):
    cols = ["run_id", "scenario_id"]
    head = pd.read_parquet(part, columns=cols)
    if head.empty:
        return None, None
    run_id = head["run_id"].iloc[0]
    scenario_id = head["scenario_id"].iloc[0]
    return run_id, scenario_id


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", default="dataset", help="Dir with part-*.parquet")
    ap.add_argument("--out", default="dataset_split")
    ap.add_argument("--mode", default="run", choices=["run", "config"])
    ap.add_argument("--ratios", type=float, nargs=3, default=[0.7, 0.15, 0.15],
                    metavar=("TRAIN", "VAL", "TEST"))
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    parts = sorted(glob.glob(os.path.join(args.dataset, "part-*.parquet")))
    if not parts:
        sys.exit(f"No part-*.parquet in {args.dataset}. Run build_dataset.py first.")

    # map each part to its grouping key
    part_keys = {}
    keyset = set()
    for p in parts:
        run_id, scenario_id = key_of(p, args.mode)
        if run_id is None:
            continue
        key = scenario_id if args.mode == "config" else run_id
        part_keys[p] = key
        keyset.add(key)

    keys = sorted(keyset)
    rng = np.random.default_rng(args.seed)
    rng.shuffle(keys)
    n = len(keys)
    n_tr = int(round(args.ratios[0] * n))
    n_va = int(round(args.ratios[1] * n))
    split_map = {}
    for i, k in enumerate(keys):
        split_map[k] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")

    counts = {"train": 0, "val": 0, "test": 0}
    os.makedirs(args.out, exist_ok=True)
    for p, key in part_keys.items():
        split = split_map[key]
        counts[split] += 1
        df = pd.read_parquet(p)
        df["split"] = split
        df.to_parquet(os.path.join(args.out, os.path.basename(p)), index=False)

    print(f"Split by {args.mode}: {n} groups -> "
          f"train={n_tr} val={n_va} test={n - n_tr - n_va}")
    print(f"Part files per split: {counts}")
    print(f"Wrote split parts to {args.out}/ (added 'split' column)")


if __name__ == "__main__":
    main()


Writing scripts/split_dataset.py


In [13]:
%%writefile scenario_matrix.yaml
# =====================================================================
# TRUST-MANET scenario matrix
# =====================================================================
# Two kinds of factor:
#   * sim_factors  -> drive distinct ns-3 runs (expensive)
#   * post_factors -> applied in Python on top of each ns-3 run, with NO
#                     re-simulation (corrupt-reporter / bad-mouthing noise is
#                     generated during feature building, so one simulation is
#                     reused for every corrupt_reporter_frac).
#
# Attack + compromised_frac are bundled so we don't waste runs on redundant
# combinations (clean implies 0% compromised; blackhole/grayhole at 0% would
# just be clean).
# =====================================================================

full:
  sim_factors:
    n_nodes: [25, 50, 100]
    speed:   [0, 5, 15]            # m/s (0 = static)
    attack:
      - {attack_type: clean,     compromised_frac: 0.00}
      - {attack_type: blackhole, compromised_frac: 0.10}
      - {attack_type: blackhole, compromised_frac: 0.20}
      - {attack_type: grayhole,  compromised_frac: 0.10}
      - {attack_type: grayhole,  compromised_frac: 0.20}
  seeds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  post_factors:
    corrupt_reporter_frac: [0.00, 0.25, 0.40]
  # -> 3 x 3 x 5 x 10 = 450 ns-3 runs; x3 corrupt fracs in post = 1350 logical scenarios.

# A predeclared fractional-factorial screening set for a fast first pass /
# pipeline validation (36 ns-3 runs).
screening:
  sim_factors:
    n_nodes: [25, 50]
    speed:   [0, 15]
    attack:
      - {attack_type: clean,     compromised_frac: 0.00}
      - {attack_type: blackhole, compromised_frac: 0.20}
      - {attack_type: grayhole,  compromised_frac: 0.20}
  seeds: [1, 2, 3]
  post_factors:
    corrupt_reporter_frac: [0.00, 0.40]
  # -> 2 x 2 x 3 x 3 = 36 ns-3 runs.

# Fixed ns-3 parameters (shared by both matrices).
fixed:
  sim_time: 200          # seconds
  window_sec: 5          # measurement window
  attack_start: 20       # attacks begin after routes have formed
  app_start: 2
  range_m: 120           # observation / 1-hop adjacency range
  grayhole_drop_prob: 0.5
  n_flows_per_25_nodes: 15     # scaled linearly with node count
  density_m2_per_node: 30000   # area = n_nodes * this; side = sqrt(area)

# Report / observation model (used only in build_dataset.py).
report_model:
  observation_sigma: 0.08      # gaussian noise on a watchdog's forward-ratio estimate
  min_transit_for_obs: 3       # min transit packets for a usable observation in a window
  badmouth_risk_min: 0.70      # corrupt reporter inflates benign target's risk to >= this
  ballotstuff_risk_max: 0.30   # corrupt reporter deflates malicious target's risk to <= this


Writing scenario_matrix.yaml


In [14]:
%%writefile requirements.txt
pandas>=2.0
numpy>=1.24
pyarrow>=12.0
pyyaml>=6.0


Writing requirements.txt


## 4. Build ns-3 (pinned ns-3.42, minimal modules) — into /kaggle/temp

~8–15 min on Kaggle's 4 cores. **If the build fails, the real error is in THIS
output** (the smoke test below only shows the symptom).

> C++20 / compiler error? `!apt-get install -y g++-12` then
> `import os; os.environ['CXX']='g++-12'` before rebuilding. Most likely tweak
> spot is `malicious-routing.*`.

In [15]:
import time; t0=time.time()
!bash scripts/build_ns3.sh "$NS3_DIR" ns-3.42
print(f"\nbuild step finished in {time.time()-t0:.0f}s")
!echo '--- scratch binary ---'; ls -la "$NS3_DIR"/build/scratch/trust-manet/ 2>/dev/null || ls -la "$NS3_DIR"/build/scratch/ 2>/dev/null


==> Checking build prerequisites
==> Cloning ns-3 (ns-3.42) into /kaggle/temp/ns-3-dev
Cloning into '/kaggle/temp/ns-3-dev'...
remote: Enumerating objects: 233240, done.
remote: Counting objects: 100% (2357/2357), done.
remote: Compressing objects: 100% (1137/1137), done.
remote: Total 233240 (delta 1360), reused 1418 (delta 1218), pack-reused 230883 (from 4)
Receiving objects: 100% (233240/233240), 212.34 MiB | 31.30 MiB/s, done.
Resolving deltas: 100% (190157/190157), done.
==> Checking out ns-3.42
Note: switching to 'ns-3.42'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by s

## 5. Smoke test — one run (only after the build above succeeded)

In [16]:
!cd "$NS3_DIR" && ./ns3 run "trust-manet --nNodes=25 --attackType=blackhole --simTime=60 --outPrefix=/kaggle/temp/smoke"
!echo '--- meta ---'; cat /kaggle/temp/smoke_meta.json
!echo '--- nodes head ---'; head -3 /kaggle/temp/smoke_nodes.csv


[0/2] Re-checking globbed directories...
ninja: no work to do.
--- meta ---
{
  "n_nodes": 25,
  "speed": 5,
  "compromised_frac": 0.1,
  "attack_type": "blackhole",
  "grayhole_drop_prob": 0.5,
  "n_flows": 15,
  "sim_time": 60,
  "window_sec": 5,
  "attack_start": 20,
  "app_start": 2,
  "area_x": 1500,
  "area_y": 1500,
  "range_m": 120,
  "seed": 1,
  "malicious_nodes": [11, 23]
}
--- nodes head ---
window_idx,time_s,node,pos_x,pos_y,vel_x,vel_y,speed,recv_forward,fwd_ok,fwd_drop_attack,no_route,app_tx,app_rx
0,5,0,18.6498,36.5451,2.97119,4.02145,5,0,0,0,0,0,0
0,5,1,168.425,293.636,4.65004,-1.83769,5,0,0,0,0,11,0


## 6. Run the SCREENING matrix (36 runs). `--resume` continues later.

In [17]:
!python scripts/run_matrix.py --ns3-dir "$NS3_DIR" --matrix scenario_matrix.yaml \
    --which screening --out "$RAW_DIR" --resume


36 runs enumerated -> /kaggle/temp/raw/run_index.csv
[1/36] n25_s0_clean_c0.00_seed1
[2/36] n25_s0_clean_c0.00_seed2
[3/36] n25_s0_clean_c0.00_seed3
[4/36] n25_s0_blackhole_c0.20_seed1
[5/36] n25_s0_blackhole_c0.20_seed2
[6/36] n25_s0_blackhole_c0.20_seed3
[7/36] n25_s0_grayhole_c0.20_seed1
[8/36] n25_s0_grayhole_c0.20_seed2
[9/36] n25_s0_grayhole_c0.20_seed3
[10/36] n25_s15_clean_c0.00_seed1
[11/36] n25_s15_clean_c0.00_seed2
[12/36] n25_s15_clean_c0.00_seed3
[13/36] n25_s15_blackhole_c0.20_seed1
[14/36] n25_s15_blackhole_c0.20_seed2
[15/36] n25_s15_blackhole_c0.20_seed3
[16/36] n25_s15_grayhole_c0.20_seed1
[17/36] n25_s15_grayhole_c0.20_seed2
[18/36] n25_s15_grayhole_c0.20_seed3
[19/36] n50_s0_clean_c0.00_seed1
[20/36] n50_s0_clean_c0.00_seed2
[21/36] n50_s0_clean_c0.00_seed3
[22/36] n50_s0_blackhole_c0.20_seed1
[23/36] n50_s0_blackhole_c0.20_seed2
[24/36] n50_s0_blackhole_c0.20_seed3
[25/36] n50_s0_grayhole_c0.20_seed1
[26/36] n50_s0_grayhole_c0.20_seed2
[27/36] n50_s0_grayhole_c0.20

## 7. Build the dataset (Parquet parts, one per run)

In [18]:
!python scripts/build_dataset.py --raw "$RAW_DIR" --out "$DS_DIR" \
    --matrix scenario_matrix.yaml --which screening


Building dataset from 36 runs x 2 corrupt fracs
  wrote /kaggle/working/dataset/part-n25_s0_blackhole_c0.20_seed1.parquet  rows=1560
  wrote /kaggle/working/dataset/part-n25_s0_blackhole_c0.20_seed2.parquet  rows=2808
  wrote /kaggle/working/dataset/part-n25_s0_blackhole_c0.20_seed3.parquet  rows=2340
  wrote /kaggle/working/dataset/part-n25_s0_clean_c0.00_seed1.parquet  rows=1560
  wrote /kaggle/working/dataset/part-n25_s0_clean_c0.00_seed2.parquet  rows=2808
  wrote /kaggle/working/dataset/part-n25_s0_clean_c0.00_seed3.parquet  rows=2340
  wrote /kaggle/working/dataset/part-n25_s0_grayhole_c0.20_seed1.parquet  rows=1560
  wrote /kaggle/working/dataset/part-n25_s0_grayhole_c0.20_seed2.parquet  rows=2808
  wrote /kaggle/working/dataset/part-n25_s0_grayhole_c0.20_seed3.parquet  rows=2340
  wrote /kaggle/working/dataset/part-n25_s15_blackhole_c0.20_seed1.parquet  rows=3856
  wrote /kaggle/working/dataset/part-n25_s15_blackhole_c0.20_seed2.parquet  rows=3800
  wrote /kaggle/working/datase

## 8. Leakage-safe split (test = unseen scenario configs)

In [19]:
!python scripts/split_dataset.py --dataset "$DS_DIR" --out "$SPLIT_DIR" --mode config


Split by config: 12 groups -> train=8 val=2 test=2
Part files per split: {'train': 24, 'val': 6, 'test': 6}
Wrote split parts to /kaggle/working/dataset_split/ (added 'split' column)


## 9. Peek + merge to a single file

On Kaggle, merging the per-run parts into **one** Parquet keeps you well under the
~500-file cap on `/kaggle/working` and is easier to download.

In [20]:
import glob, pandas as pd, pyarrow.dataset as ds
parts = glob.glob(f'{SPLIT_DIR}/*.parquet') or glob.glob(f'{DS_DIR}/*.parquet')
print('part files:', len(parts))
df = ds.dataset(parts, format='parquet').head(2000).to_pandas()
print('columns:', list(df.columns))
print(df[['scenario_id','observer_id','neighbor_id','observed_fwd_ratio',
          'reported_risk','tgt_malicious','is_false_accusation']].head())

# merge to one file (fine for screening; for the full matrix this can be large)
merged = f'{DS_DIR}/trust_manet_dataset.parquet'
pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True).to_parquet(merged, index=False)
print('merged ->', merged)


part files: 36
columns: ['window_idx', 'observer_id', 'neighbor_id', 'dist', 'rel_speed', 'neighbor_churn', 'contact_lifetime_win', 'contact_lifetime_s', 'rssi_proxy_dbm', 'link_quality_proxy', 'observed_fwd_ratio', 'obs_confidence', 'missing_ack_ratio', 'tgt_transit_load', 'tgt_no_route', 'tgt_malicious', 'attack_type', 'tgt_drop_attack_true', 'route_reachability', 'run_id', 'scenario_id', 'seed', 'n_nodes', 'speed', 'compromised_frac', 'time_s', 'corrupt_reporter_frac', 'observer_is_corrupt', 'reported_risk', 'report_age_s', 'has_report', 'observer_consistency', 'rep_count_others', 'mean_reported_risk_others', 'mean_report_age_others', 'mean_observer_consistency_others', 'is_false_accusation', 'split']
              scenario_id  observer_id  neighbor_id  observed_fwd_ratio  \
0  n25_s0_blackhole_c0.20            3           12                 NaN   
1  n25_s0_blackhole_c0.20            3           12                 NaN   
2  n25_s0_blackhole_c0.20            3           12          

In [21]:
   import shutil, os
   # working ke andar temp folder mein copy karke zip, phir working mein rakho
   shutil.make_archive('/kaggle/tmp_working', 'zip', '/kaggle/working')
   shutil.move('/kaggle/tmp_working.zip', '/kaggle/working/working_backup.zip')
   print("ready:", round(os.path.getsize('/kaggle/working/working_backup.zip')/1e6,1), "MB")

ready: 10.4 MB


In [22]:
import shutil
shutil.make_archive('/kaggle/working/dataset', 'zip', '/kaggle/working/dataset_split')

'/kaggle/working/dataset.zip'

## 10. Save, full matrix, and reusing the build across sessions

**Save the outputs:** click **Save Version** (top-right). Everything in
`/kaggle/working` (your dataset) is kept as the notebook's output. To reuse it in
another notebook, *Add Data → Your Work → this notebook's output*, or create a
Kaggle Dataset from it.

**Full matrix (450 runs)** — run in chunks across sessions (`--resume` skips done runs):
```python
!python scripts/run_matrix.py --ns3-dir "$NS3_DIR" --which full --out "$RAW_DIR" --resume
!python scripts/build_dataset.py --raw "$RAW_DIR" --out "$DS_DIR" --which full
!python scripts/split_dataset.py --dataset "$DS_DIR" --out "$SPLIT_DIR" --mode config
```
Because `/kaggle/temp` is wiped at session end, persist what you need as single
tar files under `/kaggle/working` (one file each beats the 500-file cap), then save
the version:
```python
!tar -C /kaggle/temp -czf /kaggle/working/ns3-build.tgz ns-3-dev   # cache the build
!tar -C /kaggle/temp -czf /kaggle/working/raw.tgz raw              # cache raw logs
```
Next session, add this notebook's output as input and restore instead of rebuilding:
```python
!tar -C /kaggle/temp -xzf /kaggle/input/<your-dataset>/ns3-build.tgz
!tar -C /kaggle/temp -xzf /kaggle/input/<your-dataset>/raw.tgz
```
For big background jobs, use **Save Version → Save & Run All (Commit)** (runs up to 12 h headless).

In [23]:
!python scripts/run_matrix.py --ns3-dir "$NS3_DIR" --which full --out "$RAW_DIR" --resume
!python scripts/build_dataset.py --raw "$RAW_DIR" --out "$DS_DIR" --which full
!python scripts/split_dataset.py --dataset "$DS_DIR" --out "$SPLIT_DIR" --mode config

450 runs enumerated -> /kaggle/temp/raw/run_index.csv
[4/450] n25_s0_clean_c0.00_seed4
[5/450] n25_s0_clean_c0.00_seed5
[6/450] n25_s0_clean_c0.00_seed6
[7/450] n25_s0_clean_c0.00_seed7
[8/450] n25_s0_clean_c0.00_seed8
[9/450] n25_s0_clean_c0.00_seed9
[10/450] n25_s0_clean_c0.00_seed10
[11/450] n25_s0_blackhole_c0.10_seed1
[12/450] n25_s0_blackhole_c0.10_seed2
[13/450] n25_s0_blackhole_c0.10_seed3
[14/450] n25_s0_blackhole_c0.10_seed4
[15/450] n25_s0_blackhole_c0.10_seed5
[16/450] n25_s0_blackhole_c0.10_seed6
[17/450] n25_s0_blackhole_c0.10_seed7
[18/450] n25_s0_blackhole_c0.10_seed8
[19/450] n25_s0_blackhole_c0.10_seed9
[20/450] n25_s0_blackhole_c0.10_seed10
[24/450] n25_s0_blackhole_c0.20_seed4
[25/450] n25_s0_blackhole_c0.20_seed5
[26/450] n25_s0_blackhole_c0.20_seed6
[27/450] n25_s0_blackhole_c0.20_seed7
[28/450] n25_s0_blackhole_c0.20_seed8
[29/450] n25_s0_blackhole_c0.20_seed9
[30/450] n25_s0_blackhole_c0.20_seed10
[31/450] n25_s0_grayhole_c0.10_seed1
[32/450] n25_s0_grayhole_c0.